# DoubleML DR-DiD — B2_sweep_serial (N=400)

**Workstream B2 sample-size sweep — one N per notebook** (DoubleML random-forest
nuisances at large N are slow). Loops linearity_degree=1/2/3 at this N; combine
the four N CSVs afterwards for the sqrt(N) comparison.

> **Colab:** upload just this notebook and *Run all*.


In [ ]:
# ===== Parameters (edit me) =====
SCENARIO  = "B2_sweep_serial"
N         = 400     # this notebook's sample size (200/400/800/1600)
REPS      = 100     # replications per linearity degree (matches the DiD-BCF runs)
NUM_TREES = 500     # ranger trees for the DoubleML nuisances (100 ≈ 4x faster)
DATA_JOBS = 2       # parallel workers for the (cheap) data-generation step

# ===== Install R + the DoubleML stack as fast *binary* packages =====
import shutil, subprocess, os
if shutil.which('Rscript') is None:
    subprocess.run('sudo apt-get -qq update && sudo apt-get -qq install -y r-base',
                   shell=True, check=True)
codename = (subprocess.run(['bash','-lc','. /etc/os-release && echo $VERSION_CODENAME'],
            capture_output=True, text=True).stdout.strip() or 'jammy')
r_install = r'''
options(repos = c(CRAN = sprintf('https://packagemanager.posit.co/cran/__linux__/%s/latest', Sys.getenv('PPM_CODENAME'))),
        HTTPUserAgent = sprintf('R/%s R (%s)', getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c('did','DoubleML','mlr3','mlr3learners','ranger','lgr','progress','openxlsx')
need <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(need)) install.packages(need)
cat('R packages ready:', paste(pkgs[pkgs %in% rownames(installed.packages())], collapse=', '), '\n')
'''
res = subprocess.run(['Rscript','-e', r_install], capture_output=True, text=True,
                     env={**os.environ, 'PPM_CODENAME': codename})
print(res.stdout[-3000:]); print(res.stderr[-1500:])


In [ ]:
# ===== Clone the engine + regenerate the seeded panels (sweep -> N subdirs) =====
import os, glob, subprocess
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
if not os.path.isdir("DiD-BCF"):
    subprocess.run(["git","clone","--depth","1",REPO_URL], check=True)
ROOT = "DiD-BCF/Simulation_Studies_Revision"
%pip install -q numpy pandas joblib
# --all-N writes the full sweep under R_code/<scn>_datasets/N=<N>/linearity_degree=d/
subprocess.run(f"cd {ROOT} && python DGPs/data_creation_B2_sweep_serial.py --all-N --reps {REPS} --jobs {DATA_JOBS}", shell=True, check=True)

basedir = f"{ROOT}/R_code/B2_sweep_serial_datasets"
folder  = f"{basedir}/N=400"   # this notebook reads only this sample size
# zero-pad iteration filenames in the N subdir (alphabetical order == rep order)
for ld in glob.glob(f"{folder}/linearity_degree=*"):
    for fp in glob.glob(f"{ld}/iteration_*.csv"):
        n = int(os.path.basename(fp).split("_")[1].split(".")[0])
        new = os.path.join(ld, f"iteration_{n:04d}.csv")
        if fp != new: os.rename(fp, new)
print("panels ready (zero-padded) for B2_sweep_serial N=400")


In [ ]:
# ===== Run DoubleML at this N (loops linearity_degree=1/2/3) =====
# script lives in basedir; run it with the N subdir as CWD so its relative
# 'linearity_degree=d' paths resolve to N=<N>/linearity_degree=d.
if NUM_TREES != 500:
    subprocess.run(f"sed -i 's/num.trees = 500/num.trees = {NUM_TREES}/g' {basedir}/DoubleML_did.R", shell=True, check=True)
subprocess.run(f"cd {folder} && Rscript ../DoubleML_did.R", shell=True, check=True)
print(open(f"{folder}/output_DoubleML_did.txt").read())


In [ ]:
# ===== Inspect + download the results (DiD-BCF-schema summaries) =====
import glob, sys, subprocess, os, pandas as pd
from IPython.display import display
sys.path.insert(0, ROOT)
from did_bcf_revision.metrics import compute_metrics, surface_metrics
# tag the N-subdir outputs with N so the four sweeps don't collide when combined
for d in (1, 2, 3):
    src = f"{folder}/summaries_doubleml_B2_sweep_serial_lin_{d}.csv"
    dst = f"{folder}/summaries_doubleml_B2_sweep_serial_N400_lin_{d}.csv"
    if os.path.exists(src): os.replace(src, dst)
csvs = sorted(glob.glob(f"{folder}/summaries_doubleml_B2_sweep_serial_N400_lin_*.csv"))
for f in csvs: print(f)
if csvs:
    summ = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    print("\nDecomposed metrics (compute_metrics):"); display(compute_metrics(summ))
    print("\nCATT-surface metrics (surface_metrics):"); display(surface_metrics(summ))
else:
    print("No summaries written -- check the run-cell output above.")
zipname = "DoubleML_B2_sweep_serial_N400_results.zip"
subprocess.run(f"cd {folder} && zip -q {zipname} summaries_doubleml_B2_sweep_serial_N400_lin_*.csv output_DoubleML_did.txt", shell=True)
try:
    from google.colab import files
    files.download(f"{folder}/{zipname}")
except Exception as e:
    print("(not on Colab / download skipped):", e)
